### Ноутбук, в котором генерируются фичи с использованием OpenFE

In [ ]:
import pandas as pd

from ml_pipeline.classification.config import classification_config as config
from ml_pipeline.base.utils import set_seed

In [2]:
set_seed(config.general.seed)

In [3]:
full_df = pd.read_csv(config.paths.train, index_col="PassengerId")
full_test_df = pd.read_csv(config.paths.test, index_col="PassengerId")
full_df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Логистическая регрессия - просто как модель, из параметров которой можно взять список препроцессоров

По сути тут используется конфиг пайплайна без самого пайплайна

In [4]:
config_logreg = config.copy()

model_name = "logistic_regression"

config_logreg.experiment.to_train = [
    {
        "model": model_name
    }
]

config_logreg.split.cv = False

In [5]:
from ml_pipeline.utils import build_transformers, apply_transformers

model_config = config_logreg.models.get(model_name)

target = config_logreg.data.target_col

y = full_df[target].values
X_train_raw = full_df.drop(columns=[target])

preprocess_transformers = build_transformers(model_config, config_logreg)

X_train = apply_transformers(preprocess_transformers, X_train_raw, fit=True)
X_test = apply_transformers(preprocess_transformers, full_test_df, fit=False)

In [7]:
from openfe import OpenFE, transform

ofe = OpenFE()

features = ofe.fit(data=X_train, label=y, n_jobs=4)

The number of candidate features is 1449
Start stage I selection.


100%|██████████| 16/16 [00:04<00:00,  3.26it/s]


1147 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|██████████| 16/16 [00:03<00:00,  4.95it/s]


The number of remaining candidate features is 301
Start stage II selection.


100%|██████████| 16/16 [00:02<00:00,  6.31it/s]


Finish data processing.


In [ ]:
ofe_train_x, ofe_test_x = transform(X_train, X_test, features, n_jobs=4) # transform the train and test data according to generated features.

Поменять название индекса и влить таргет в один датафрейм

In [ ]:
ofe_train_x.index.name = 'PassengerId'
ofe_train_x['Survived'] = y
ofe_train_x

C:\Users\az\AppData\Local\Temp\ipykernel_21696\4170147582.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_x['Survived'] = y


,Pclass,Sex,SibSp,Parch,Family_Size,Alone,More_Than_4_relatives,Embarked_C,Embarked_S,Age_Group,Fare_Range,Ticket_Type,autoFE_f_0,autoFE_f_1,autoFE_f_2,autoFE_f_3,autoFE_f_4,autoFE_f_5,autoFE_f_6,autoFE_f_7,autoFE_f_8,autoFE_f_9,autoFE_f_10,autoFE_f_11,autoFE_f_12,autoFE_f_13,autoFE_f_14,autoFE_f_15,autoFE_f_16,autoFE_f_17,autoFE_f_18,autoFE_f_19,autoFE_f_20,autoFE_f_21,autoFE_f_22,autoFE_f_23,autoFE_f_24,autoFE_f_25,autoFE_f_26,autoFE_f_27,...,autoFE_f_262,autoFE_f_263,autoFE_f_264,autoFE_f_265,autoFE_f_266,autoFE_f_267,autoFE_f_268,autoFE_f_269,autoFE_f_270,autoFE_f_271,autoFE_f_272,autoFE_f_273,autoFE_f_274,autoFE_f_275,autoFE_f_276,autoFE_f_277,autoFE_f_278,autoFE_f_279,autoFE_f_280,autoFE_f_281,autoFE_f_282,autoFE_f_283,autoFE_f_284,autoFE_f_285,autoFE_f_286,autoFE_f_287,autoFE_f_288,autoFE_f_289,autoFE_f_290,autoFE_f_291,autoFE_f_292,autoFE_f_293,autoFE_f_294,autoFE_f_295,autoFE_f_296,autoFE_f_297,autoFE_f_298,autoFE_f_299,autoFE_f_300,Survived
PassengerId,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,3,0,1,0,1,0,0,0.0,1.0,2,1,0.810458,0.476275,NaN,1.620915,0.810458,3.810458,0.700000,0.310106,17.0,0.848936,0.810458,346.0,2.0,0.873061,0.810458,0.846806,118.0,640.0,1.810458,-0.189542,0.405229,-0.189542,0.0,247.0,0.832288,0.839130,0.810458,1.810458,-1.189542,...,0.810458,1.000000,2.559880,0.167842,0.157996,0.163478,6.0,79.0,0.484170,0.501567,2.0,0.523404,0.810458,0.482609,0.772640,0.810458,0.391396,2.0,0.324775,0.402695,8.0,0.000000,0.0,0.500705,493.0,0.425217,3.0,1.000000,0.500000,0.147094,1.0,0.722144,1.000000,0.846373,0.837485,359.0,2.0,3.0,0.420000,0
2,1,1,1,0,1,0,0,1.0,0.0,3,1,0.895425,0.457082,1.0,2.686275,-0.104575,1.895425,0.154334,0.721679,60.0,0.917021,0.895425,145.0,3.0,0.899381,1.000000,0.910679,111.0,312.0,1.895425,0.895425,0.298475,-0.104575,1.0,272.0,0.920063,0.922833,0.895425,1.895425,-2.104575,...,0.000000,1.000000,2.559880,0.253870,0.157996,0.164905,3.0,104.0,0.454198,0.501567,3.0,0.523404,0.895425,0.473573,0.157996,0.895425,0.376161,3.0,0.324775,0.902695,3.0,0.000000,1.0,0.397833,144.0,0.420719,1.0,1.000000,0.333333,0.292918,1.0,0.657895,1.000000,0.846373,0.920601,160.0,3.0,2.0,0.883721,1
3,3,1,0,0,0,1,0,0.0,1.0,2,1,0.967320,0.457082,3.0,1.934641,-0.032680,3.967320,0.700000,0.310106,18.0,0.985443,0.967320,346.0,2.0,0.978138,0.967320,0.985529,420.0,640.0,1.967320,-0.032680,0.483660,0.967320,0.0,194.0,0.985971,0.977391,NaN,0.967320,-1.032680,...,0.967320,0.967320,2.559880,0.667842,0.000000,0.663478,6.0,472.0,0.484170,0.294052,2.0,0.245570,0.967320,0.482609,0.701899,0.967320,0.391396,0.0,0.281902,0.402695,8.0,0.967320,1.0,0.500705,216.0,0.425217,NaN,0.967320,0.000000,0.792918,1.0,0.333568,0.000000,0.361858,0.980687,557.0,2.0,4.0,0.420000,1
4,1,1,1,0,1,0,0,0.0,1.0,3,1,0.019608,0.457082,1.0,0.058824,-0.980392,1.019608,0.154334,0.721679,66.0,0.057447,0.019608,145.0,3.0,0.212074,0.019608,0.052894,111.0,312.0,1.019608,-0.980392,0.006536,-0.980392,1.0,272.0,0.048589,0.070825,0.019608,1.019608,-2.980392,...,0.019608,1.000000,2.559880,0.253870,0.157996,0.164905,3.0,104.0,0.484170,0.501567,3.0,0.523404,0.019608,0.473573,0.157996,0.019608,0.376161,3.0,0.324775,0.402695,3.0,0.000000,1.0,0.397833,144.0,0.420719,1.0,1.000000,0.333333,0.292918,1.0,0.657895,1.000000,0.846373,0.051502,359.0,3.0,2.0,0.383721,1
5,3,0,0,0,0,1,0,0.0,1.0,3,1,0.633987,0.476275,NaN,1.901961,0.633987,3.633987,0.738901,0.721679,4.0,0.708228,0.633987,248.0,3.0,0.711566,0.633987,0.709082,350.0,640.0,1.633987,-0.366013,0.211329,0.633987,0.0,596.0,0.705387,0.695560,NaN,0.633987,-2.366013,...,0.633987,0.633987,2.559880,0.667842,0.000000,0.664905,9.0,472.0,0.484170,0.294052,3.0,0.245570,0.633987,0.473573,0.701899,0.633987,0.391396,0.0,0.281902,0.402695,8.0,0.633987,0.0,0.500705,493.0,0.420719,NaN,0.633987,0.000000,0.647094,1.0,0.333568,0.000000,0.361858,0.692764,557.0,3.0,3.0,0.383721,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [ ]:
ofe_test_x

,Pclass,Sex,SibSp,Parch,Family_Size,Alone,More_Than_4_relatives,Embarked_C,Embarked_S,Age_Group,Fare_Range,Ticket_Type,autoFE_f_0,autoFE_f_1,autoFE_f_2,autoFE_f_3,autoFE_f_4,autoFE_f_5,autoFE_f_6,autoFE_f_7,autoFE_f_8,autoFE_f_9,autoFE_f_10,autoFE_f_11,autoFE_f_12,autoFE_f_13,autoFE_f_14,autoFE_f_15,autoFE_f_16,autoFE_f_17,autoFE_f_18,autoFE_f_19,autoFE_f_20,autoFE_f_21,autoFE_f_22,autoFE_f_23,autoFE_f_24,autoFE_f_25,autoFE_f_26,autoFE_f_27,...,autoFE_f_261,autoFE_f_262,autoFE_f_263,autoFE_f_264,autoFE_f_265,autoFE_f_266,autoFE_f_267,autoFE_f_268,autoFE_f_269,autoFE_f_270,autoFE_f_271,autoFE_f_272,autoFE_f_273,autoFE_f_274,autoFE_f_275,autoFE_f_276,autoFE_f_277,autoFE_f_278,autoFE_f_279,autoFE_f_280,autoFE_f_281,autoFE_f_282,autoFE_f_283,autoFE_f_284,autoFE_f_285,autoFE_f_286,autoFE_f_287,autoFE_f_288,autoFE_f_289,autoFE_f_290,autoFE_f_291,autoFE_f_292,autoFE_f_293,autoFE_f_294,autoFE_f_295,autoFE_f_296,autoFE_f_297,autoFE_f_298,autoFE_f_299,autoFE_f_300
PassengerId,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
892,3,0,0,0,0,1,0,0.0,0.0,3,1,0.379085,0.476275,NaN,1.137255,0.379085,3.379085,0.738901,0.721679,13.0,0.346203,0.379085,248.0,3.0,0.181946,0.379085,0.368762,350.0,640.0,1.379085,0.379085,0.126362,0.379085,0.0,596.0,0.358586,0.393235,NaN,0.379085,-2.620915,...,1.0,0.000000,0.379085,2.559880,0.667842,0.000000,0.664905,9.0,472.0,0.454198,0.294052,3.0,0.245570,0.379085,0.473573,0.701899,0.379085,0.391396,0.0,0.281902,0.402695,8.0,0.379085,0.0,0.500705,493.0,0.420719,NaN,0.379085,0.000000,0.647094,1.0,0.333568,0.000000,0.361858,0.353499,233.0,3.0,3.0,0.383721
893,3,1,1,0,1,0,0,0.0,1.0,3,1,0.464052,0.457082,3.0,1.392157,-0.535948,3.464052,0.738901,0.721679,3.0,0.536170,0.464052,248.0,3.0,0.331453,0.464052,0.475050,111.0,640.0,1.464052,-0.535948,0.154684,-0.535948,0.0,272.0,0.529781,0.479915,0.464052,1.464052,-2.535948,...,0.0,0.464052,1.000000,2.559880,0.167842,0.157996,0.164905,9.0,79.0,0.484170,0.501567,3.0,0.523404,0.464052,0.473573,0.772640,0.464052,0.391396,3.0,0.324775,0.402695,8.0,0.000000,1.0,0.500705,216.0,0.420719,3.0,1.000000,0.333333,0.292918,1.0,0.722144,1.000000,0.846373,0.482833,359.0,3.0,4.0,0.383721
894,2,0,0,0,0,1,0,0.0,0.0,4,1,0.209150,0.476275,NaN,0.836601,0.209150,2.209150,0.787736,0.948248,5.0,0.174684,0.209150,24.0,4.0,0.144404,0.209150,0.199601,64.0,275.0,1.209150,0.209150,0.052288,0.209150,2.0,596.0,0.175084,0.452830,NaN,0.209150,-3.790850,...,1.0,0.000000,0.209150,2.559880,0.716606,0.000000,0.750000,8.0,158.0,0.454198,0.294052,4.0,0.245570,0.209150,0.438679,0.303165,0.209150,0.373646,0.0,0.281902,0.402695,3.0,0.209150,0.0,0.501805,171.0,0.377358,NaN,0.209150,0.000000,0.647094,1.0,0.287004,0.000000,0.361858,0.179715,233.0,4.0,2.0,0.349057
895,3,0,0,0,0,1,0,0.0,1.0,2,1,0.359477,0.476275,NaN,0.718954,0.359477,3.359477,0.700000,0.310106,11.0,0.329747,0.359477,346.0,2.0,0.165726,0.359477,0.354291,420.0,640.0,1.359477,-0.640523,0.179739,0.359477,0.0,596.0,0.343996,0.315652,NaN,0.359477,-1.640523,...,1.0,0.359477,0.359477,2.559880,0.667842,0.000000,0.663478,6.0,472.0,0.484170,0.294052,2.0,0.245570,0.359477,0.482609,0.701899,0.359477,0.391396,0.0,0.281902,0.402695,8.0,0.359477,0.0,0.500705,493.0,0.425217,NaN,0.359477,0.000000,0.647094,1.0,0.333568,0.000000,0.361858,0.343416,557.0,2.0,3.0,0.420000
896,3,1,1,1,2,0,0,0.0,1.0,2,1,0.352941,0.457082,3.0,0.705882,-0.647059,3.352941,0.700000,0.310106,6.0,0.440252,-0.647059,346.0,2.0,0.155148,0.352941,0.429412,118.0,640.0,1.352941,-0.647059,0.176471,-1.647059,0.0,272.0,0.446708,0.303478,0.176471,1.352941,-1.647059,...,0.0,0.352941,1.000000,2.094118,0.167842,0.157996,0.163478,6.0,75.0,0.484170,0.501567,2.0,0.496855,0.352941,0.482609,0.772640,1.000000,0.836389,4.0,0.324775,0.352941,8.0,0.000000,1.0,0.500705,216.0,0.891304,3.0,2.000000,1.000000,0.292918,1.0,0.830748,1.414214,0.846373,0.379828,359.0,2.0,4.0,0.420000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

Сохранить в csv

In [ ]:
ofe_train_x.to_csv(f"data/train_openfe.csv", index=True)

In [ ]:
ofe_test_x.to_csv(f"data/test_openfe.csv", index=True)